# Multivariate non-stationary modelling with VGP

In this notebook we will see how VGP can be used to model multiple variables of different types and how they can influence each other.

In [ ]:
%%capture
!pip install cmcrameri # scientific color maps
!pip install git+https://github.com/italo-goncalves/geoML.git@claude

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex
from cmcrameri import cm
import os
import seaborn as sns

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import geoml
import geoml.kernels as kr
import geoml.transform as tr
import geoml.latent as gl
import geoml.warping as wp
import geoml.likelihood as lk

The Jura dataset:

In [ ]:
jura_data, jura_validation = geoml.datasets.jura()

ELEMENTS = jura_data.variables['Elements'].labels

jura_data

## Exploratory data analysis

There is some evidence that the metal contamination in the Jura dataset might be linked to the rock type. We will make some plots to try to uncover this relationship.

In [ ]:
jura_df = jura_data.as_data_frame()
jura_df

In [ ]:
jura_df_long = jura_df.melt(
    id_vars=['Rock_measurements_a', 'Landuse_measurements_a'],
    value_vars=[f'Elements_{el}_measurements' for el in ELEMENTS],
    var_name='Element',
    value_name='Concentration'
    )
jura_df_long

In [ ]:
g = sns.FacetGrid(jura_df_long, col="Element", hue="Rock_measurements_a",
                  col_wrap=2, sharex=True, sharey=False, aspect=2)
g.map_dataframe(sns.boxplot, y="Concentration", x="Rock_measurements_a");

In [ ]:
sns.pairplot(jura_df, vars=[f'Elements_{el}_measurements' for el in ELEMENTS],
             hue='Rock_measurements_a', palette='Set1');

In [ ]:
exp_data = geoml.plots.Explorer(jura_data, continuous='Elements',
                                categorical='Rock')
exp_data.pca(0.99);

## Non-stationary mixed modelling

We will build a VGP model under the hypothesis that the elements can be influenced by rock type.

### Step #1: latent variable network

In [ ]:
# creating empty grid
grid = geoml.data.Grid2D(start=[0, 0], n=[501, 501], end=[6, 6])

# inducing points
inducing_points = geoml.data.inducing.experts(
    geoml.data.inducing.combine(
        jura_data,
        geoml.data.Grid2D(start=[0, 0], n=[21, 21], end=[6, 6])),
    4)

# network input node
net_input = gl.BasicInput(inducing_points=inducing_points,
                          transform=tr.Isotropic(0.05))

# Stochastic differential equation for rock types
field = gl.BasicGP(net_input, size=2, kernel=kr.Gaussian())
coords = gl.GPWalk(field, n_steps=5)

# The final categorical latent variables
cat = gl.BasicGP(coords, size=5, kernel=kr.Matern32())

# This node extracts information from the
# categorical variables.
# With unit_norm=False the model can zero out
# unnecessary information.
trend = gl.Linear(cat, size=7, unit_norm=False)

# The numerical variables are modelled in the same
# way as in the stationary case.
num = gl.BasicGP(net_input, size=7, kernel=kr.Spherical())

# Now the information from the categorical variables
# is used to influence the numerical ones.
num_2 = gl.LinearCombination(trend, num)

# All variables are consolidated in a single object
net_out = gl.Concatenate(cat, num_2)

### Step #2: likelihoods and normalization

The likelihood (distribution of the error) must be defined. In this case the Laplace likelihood is used to deal with the extreme values.

The likelihood includes the *warping function*, used to normalize the data to zero mean and unit variance, deal with the non-negativity constraint (these are metal concentrations) and to compensate for asymmetric distributions. A categorical likelihood for the `Rock` variable is also used.

In [ ]:
likelihoods = [
    lk.CategoricalGaussianIndicator(5),
    lk.Laplace(
        warping=wp.ChainedWarping(
            wp.Log(7), # non-negativity
            wp.RobustPCA(7, 7), # normalization
            wp.Spline(7, knots_per_arm=5),    # asymmetry
            wp.ZScore(7),
       )
    )
]

### Step #3: training

The VGP model is created and trained. It works by maximizing the Evidence Lower Bound (ELBO). Note how multiple variables can be modelled simultaneously.

In [ ]:
model = geoml.models.VGPNetwork(
    data=jura_data,
    variables=['Rock', 'Elements'],
    likelihoods=likelihoods,
    latent_network=net_out,
    options=geoml.models.GPOptions(
        prediction_batch_size=1000,
        jitter=1e-6
        )
    )
model.train_full(250)

plt.figure(figsize=(10, 5))
plt.plot(model.training_log)
plt.xlabel("Iteration")
plt.ylabel("ELBO")

### Prediction at the data points

Used later to compare true vs calculated values, as a measure of model fit.

In [ ]:
model.predict(jura_data)
model.predict(jura_validation)

### Prediction in a grid and plotting

In [ ]:
model.predict(grid, n_sim=25)

The quantiles and predictions are derived from the samples, and are thus noisy. The `sigma` parameter is used to smooth the results for plotting.

In [ ]:
# quantiles of each element's predictive distribution
jura_data.variables['Elements'].reset_quantiles([0.025, 0.5, 0.975])
jura_validation.variables['Elements'].reset_quantiles([0.025, 0.5, 0.975])
grid.variables['Elements'].reset_quantiles([0.025, 0.5, 0.975])

sigma = 1

## Results

First we take a look at the categorical variables.

In [ ]:
pred = grid.get('Rock/predicted').as_image()
uncertainty = grid.get('Rock/uncertainty').as_image()

# plotting helpers
rgb_mat = cm.roma(np.linspace(0, 1, len(jura_data.variables['Rock'].labels)))
rock_colors = [rgb2hex(c) for c in rgb_mat]

label_dict = {l: i for i, l in enumerate(jura_data.variables['Rock'].labels)}
pred_num = np.vectorize(label_dict.get)(pred)

threshold = 0.75

fig, ax = plt.subplots(figsize=(10, 10))

ax.imshow(np.where(uncertainty < threshold, pred_num, np.nan),
          cmap=cm.roma,
          vmin=0, vmax=len(rock_colors)-1,
          extent=[0, 6, 0, 6], origin='lower')

sns.scatterplot(data=jura_df, x='Xloc', y='Yloc', hue='Rock_measurements_a', ax=ax,
                hue_order=jura_data.variables['Rock'].labels,
                palette=rock_colors)

ax.set_aspect('equal')
ax.set_xlabel('X (km)')
ax.set_ylabel('Y (km)')
fig.show()

A confusion matrix to verify the accuracy on the validation data.

In [ ]:
conf = confusion_matrix(
    jura_validation.variables["Rock"].measurements_a.values,
    jura_validation.variables["Rock"].predicted.values)
conf_disp = ConfusionMatrixDisplay(
    conf, display_labels=jura_validation.variables["Rock"].labels)
conf_disp.plot()

Validation metrics:

In [ ]:
jura_validation.variables["Rock"].compute_metrics()

Numerical variables (median, confidence interval, and model fit):

In [ ]:
maxs = [np.max(jura_data.values(f'Elements/{el}/measurements')) for el in ELEMENTS]

fig, ax = plt.subplots(7, 5, figsize=(12, 20),
                       gridspec_kw={"width_ratios": [1, 1, 1, 0.025, 1],
                                    "hspace": 0.3, "wspace": 0.15})
for i, el in enumerate(ELEMENTS):
    ax[i, 0].scatter(
        jura_data.coordinates[:, 0],
        jura_data.coordinates[:, 1],
        c=jura_data.values(f'Elements/{el}/measurements'),
        cmap=cm.davos, s=2)
    ax[i, 0].set_xlim(0, 6)
    ax[i, 0].set_ylim(0, 6)
    ax[i, 0].set_aspect("equal")
    ax[i, 0].set_title(el + " - data")
    im = ax[i, 1].imshow(
        grid.get(f'Elements/{el}/quantiles/0.5').as_image(), vmin=0, vmax=maxs[i],
        origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
    ax[i, 1].set_title(el + " - median")
    ax[i, 1].set_xticks([0, 2, 4, 6])
    ax[i, 2].imshow(
        grid.get(f'Elements/{el}/quantiles/0.975').as_image(sigma=sigma*3)
        - grid.get(f'Elements/{el}/quantiles/0.025').as_image(sigma=sigma*3),
        vmin=0, vmax=maxs[i],
        origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
    ax[i, 2].set_title(el + " - 95% C.I. width")
    ax[i, 2].set_xticks([0, 2, 4, 6])

    plt.colorbar(im, cax=ax[i, 3])
    box = ax[i, 3].get_position()
    box.p0[0] -= 0.01
    box.p1[0] -= 0.01
    ax[i, 3].set_position(box)

    ax[i, 4].plot([0, maxs[i]], [0, maxs[i]], "-k")
    ax[i, 4].scatter(
        jura_data.values(f'Elements/{el}/measurements'),
        jura_data.values(f'Elements/{el}/quantiles/0.5'),
        alpha=0.2)
    ax[i, 4].set_title(el + " - real vs. predicted")
    ax[i, 4].set_aspect("equal")
    ax[i, 4].set_xlim([0, maxs[i]])
    ax[i, 4].set_ylim([0, maxs[i]])
    ax[i, 4].set_yticklabels([" "]*len(ax[i, 4].get_yticklabels()))

fig.show()

Pairs plot: how well does the model replicate the data distribution?

In [ ]:
exp_data = geoml.plots.Explorer(jura_data, continuous='Elements',
                                categorical='Rock', model=model)
exp_data.simulation_pairs();

The transformed pairs plot shows the internal representation of the variables. They should be close to independent and Gaussian-distributed.

In [ ]:
exp_data.transformed_pairs(upper='density');

Simulations:

In [ ]:
fig, ax = plt.subplots(7, 6, figsize=(12, 18),
                       gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 0.1],
                                    "hspace": 0.2, "wspace": 0.3})
for i, el in enumerate(ELEMENTS):
    for j in range(5):
        im = ax[i, j].imshow(
            grid.get(f'Elements/{el}/simulations/{j}').as_image(),
            vmin=0, vmax=maxs[i],
            origin="lower", extent=(0, 6, 0, 6), cmap=cm.davos)
        ax[i, j].set_title(el + " - simulation %d" % j)
        ax[i, j].set_xticks([0, 2, 4, 6])

        plt.colorbar(im, cax=ax[i, 5])

fig.show()

In [ ]:
exp_data.variogram();

## Validation

In [ ]:
exp_val = geoml.plots.Explorer(jura_validation, continuous='Elements',
                               categorical='Rock', model=model)

In [ ]:
exp_val.accuracy();

In [ ]:
exp_val.spread_check(bins=5);